# Unificador de archivos GRD FONASA
Este notebook lee todos los `.txt` de una carpeta, detecta automáticamente su estructura y los une en un solo dataset.

In [ ]:
import pandas as pd
import os
import chardet
from csv import Sniffer

# ──────────────────────────────────────────────
# 🔧 CONFIGURACIÓN — solo cambia esto
# ──────────────────────────────────────────────
CARPETA = r"C:\Users\jtoma\Downloads\datos grd"   # <-- pon aquí la ruta a tu carpeta
ARCHIVO_SALIDA = "dataset_grd_fonasa.csv"
# ──────────────────────────────────────────────

In [ ]:
def detectar_encoding(ruta, bytes_muestra=10000):
    """Detecta el encoding del archivo."""
    with open(ruta, 'rb') as f:
        muestra = f.read(bytes_muestra)
    resultado = chardet.detect(muestra)
    enc = resultado['encoding'] or 'utf-8'
    # Normalizar encodings comunes en archivos chilenos
    if enc.lower() in ('iso-8859-1', 'latin-1', 'windows-1252'):
        enc = 'latin-1'
    return enc

def detectar_separador(ruta, encoding):
    """Detecta el separador del archivo (coma, pipe, tab, punto y coma)."""
    with open(ruta, 'r', encoding=encoding, errors='replace') as f:
        muestra = f.read(5000)
    # Contar ocurrencias de separadores candidatos
    candidatos = [',', '|', '\t', ';']
    conteos = {sep: muestra.count(sep) for sep in candidatos}
    separador = max(conteos, key=conteos.get)
    return separador

def leer_txt(ruta):
    """Lee un TXT detectando encoding y separador automáticamente."""
    nombre = os.path.basename(ruta)
    try:
        enc = detectar_encoding(ruta)
        sep = detectar_separador(ruta, enc)
        df = pd.read_csv(ruta, sep=sep, encoding=enc, dtype=str,
                         on_bad_lines='warn', low_memory=False)
        df.columns = df.columns.str.strip()  # limpiar espacios en nombres
        df['__archivo_origen__'] = nombre    # columna para trazabilidad
        print(f"✅ {nombre:40s} | encoding: {enc:12s} | sep: {repr(sep)} | filas: {len(df):>6,} | columnas: {len(df.columns)-1}")
        return df
    except Exception as e:
        print(f"❌ {nombre:40s} | ERROR: {e}")
        return None

In [ ]:
# Listar archivos TXT en la carpeta
archivos = [os.path.join(CARPETA, f)
            for f in os.listdir(CARPETA)
            if f.lower().endswith('.txt')]

print(f"📂 Archivos TXT encontrados: {len(archivos)}\n")
for a in archivos:
    print(" -", os.path.basename(a))

In [ ]:
# Leer todos los archivos
print("Leyendo archivos...\n")
dataframes = [leer_txt(a) for a in archivos]
dataframes = [df for df in dataframes if df is not None]  # descartar errores

print(f"\n📊 Archivos leídos correctamente: {len(dataframes)}/{len(archivos)}")

In [ ]:
# Revisar columnas de cada archivo antes de unir
print("Columnas por archivo:\n")
for df in dataframes:
    nombre = df['__archivo_origen__'].iloc[0]
    cols = [c for c in df.columns if c != '__archivo_origen__']
    print(f"  {nombre}")
    print(f"    {cols}\n")

In [ ]:
# ──────────────────────────────────────────────
# 🔧 MAPEO DE COLUMNAS EQUIVALENTES
# Agrega aquí todos los nombres que representan lo mismo
# Formato: 'nombre_original' -> 'nombre_unificado'
# ──────────────────────────────────────────────
MAPEO_COLUMNAS = {
    # CIP encriptado e id paciente son lo mismo
    'cip encriptado'  : 'id_paciente',
    'cip_encriptado'  : 'id_paciente',
    'CIP Encriptado'  : 'id_paciente',
    'CIP_ENCRIPTADO'  : 'id_paciente',
    'id paciente'     : 'id_paciente',
    'id_paciente'     : 'id_paciente',
    'ID Paciente'     : 'id_paciente',
    'ID_PACIENTE'     : 'id_paciente',
    # Agrega más equivalencias aquí si las encuentras
}

def normalizar_columnas(df):
    """Renombra columnas según el mapeo definido (insensible a mayúsculas/espacios)."""
    mapeo_lower = {k.lower().strip(): v for k, v in MAPEO_COLUMNAS.items()}
    nuevos_nombres = {}
    for col in df.columns:
        clave = col.lower().strip()
        if clave in mapeo_lower:
            nuevos_nombres[col] = mapeo_lower[clave]
    if nuevos_nombres:
        archivo = df['__archivo_origen__'].iloc[0]
        for orig, nuevo in nuevos_nombres.items():
            print(f"   🔄 [{archivo}] '{orig}' → '{nuevo}'")
    return df.rename(columns=nuevos_nombres)

print("Normalizando columnas equivalentes...\n")
dataframes = [normalizar_columnas(df) for df in dataframes]
print("\n✅ Normalización lista.")

In [ ]:
# Unir todos en un solo dataset
# join='outer' → conserva TODAS las columnas de todos los archivos
# Las celdas sin valor quedan como NaN
dataset = pd.concat(dataframes, axis=0, join='outer', ignore_index=True)

# Mover columna de origen al frente
cols = ['__archivo_origen__'] + [c for c in dataset.columns if c != '__archivo_origen__']
dataset = dataset[cols]

print(f"\n✅ Dataset unificado:")
print(f"   Filas totales  : {len(dataset):,}")
print(f"   Columnas totales: {len(dataset.columns)}")
dataset.head()

In [ ]:
# Resumen de completitud por columna
completitud = (dataset.notna().sum() / len(dataset) * 100).round(1)
resumen = pd.DataFrame({
    'columna': completitud.index,
    'completitud_%': completitud.values,
    'filas_con_dato': dataset.notna().sum().values
}).sort_values('completitud_%', ascending=False)

print("Completitud por columna:")
resumen

In [ ]:
# Guardar el dataset final
ruta_salida = os.path.join(CARPETA, ARCHIVO_SALIDA)
dataset.to_csv(ruta_salida, index=False, encoding='utf-8-sig')  # utf-8-sig para compatibilidad con Excel

print(f"💾 Dataset guardado en:\n   {ruta_salida}")